In [149]:
import pandas as pd
import numpy as np

In [150]:
df = pd.read_csv("../data/Breast_Cancer.csv")

In [151]:
print("=== ORIGINAL COLUMN NAMES ===")
print("Checking for any issues before we fix them")
print(df.columns.tolist())

=== ORIGINAL COLUMN NAMES ===
Checking for any issues before we fix them
['Age', 'Race', 'Marital Status', 'T Stage ', 'N Stage', '6th Stage', 'differentiate', 'Grade', 'A Stage', 'Tumor Size', 'Estrogen Status', 'Progesterone Status', 'Regional Node Examined', 'Reginol Node Positive', 'Survival Months', 'Status']


In [152]:
# T Stage column has an invisible space at the end which will cause bugs
df = df.rename(columns={"T Stage " : "T Stage"})

# There appears to be a spelling error in ‘Reginol Node Positive' column
df = df.rename(columns={"Reginol Node Positive" : "Regional Node Positive"})

In [153]:
print("=== FIXED COLUMNS NAMES ===")
print("Trailing space and typo have been corrected")
print(df.columns.tolist())

=== FIXED COLUMNS NAMES ===
Trailing space and typo have been corrected
['Age', 'Race', 'Marital Status', 'T Stage', 'N Stage', '6th Stage', 'differentiate', 'Grade', 'A Stage', 'Tumor Size', 'Estrogen Status', 'Progesterone Status', 'Regional Node Examined', 'Regional Node Positive', 'Survival Months', 'Status']


In [154]:
# Drop the 'Survival Months' column. This feature represents how long a patient survived after diagnosis. Keeping it would cause data leakage, allowing the model to indirectly predict survival status using future information.
df = df.drop(columns=["Survival Months"])

# Drop 'differentiate' column. This column contains the same information as 'Grade' just written in words.
df = df.drop(columns=["differentiate"])

print("=== COLUMNS AFTER DROPPING UNNECESSARY ONES ===")
print("Survival Months and Differentiate have been removed")
print(df.columns.tolist())

print()
print("=== SHAPE AFTER DROPPING ===")
print("We should now have 14 columns instead of 16")
print(df.shape)

=== COLUMNS AFTER DROPPING UNNECESSARY ONES ===
Survival Months and Differentiate have been removed
['Age', 'Race', 'Marital Status', 'T Stage', 'N Stage', '6th Stage', 'Grade', 'A Stage', 'Tumor Size', 'Estrogen Status', 'Progesterone Status', 'Regional Node Examined', 'Regional Node Positive', 'Status']

=== SHAPE AFTER DROPPING ===
We should now have 14 columns instead of 16
(4024, 14)


In [155]:
# Convert Status column from text to numbers
df["Status"] = df["Status"].map({"Alive": 0, "Dead": 1})

print("=== TARGET VARIABLE AFTER ENCODING ===")
print("Alive should now be 0 and dead should now be 1")
print(df["Status"].value_counts())

=== TARGET VARIABLE AFTER ENCODING ===
Alive should now be 0 and dead should now be 1
Status
0    3408
1     616
Name: count, dtype: int64


In [156]:
# Encode T Stage - T1 is smallest tumour, T4 is largest
df["T Stage"] = df["T Stage"].map({"T1": 1, "T2": 2, "T3" : 3, "T4": 4})
print(df["T Stage"].value_counts())

T Stage
2    1786
1    1603
3     533
4     102
Name: count, dtype: int64


In [157]:
# Encode N Stage - N1 is least lymph node involvement, N3 is most
df["N Stage"] = df["N Stage"].map({"N1": 1, "N2": 2, "N3": 3, "N4": 4})
print(df["N Stage"].value_counts())

N Stage
1    2732
2     820
3     472
Name: count, dtype: int64


In [158]:
# Encode 6th Stage - cancer staging from least to most advanced#
df["6th Stage"] = df["6th Stage"].map({
    "IIA": 1,
    "IIB": 2,
    "IIIA": 3,
    "IIIB": 4,
    "IIIC": 5
})

print(df["6th Stage"].value_counts())

6th Stage
1    1305
2    1130
3    1050
5     472
4      67
Name: count, dtype: int64


In [159]:
# Encode A Stage — Regional means contained, Distant means spread to other organs
df["A Stage"] = df["A Stage"].map({"Regional": 0, "Distant": 1})
print(df["A Stage"].value_counts())

A Stage
0    3932
1      92
Name: count, dtype: int64


In [160]:
# Encode Grade - 1 is slow growing, 3 is aggressive. Grade is already numeric in the dataset but stored as object type. We convert it explicitly to integer to be safe
df["Grade"] = df["Grade"].map({" anaplastic; Grade IV": 4, "1": 1, "2": 2, "3": 3})
print(df["Grade"].value_counts())

Grade
2    2351
3    1111
1     543
4      19
Name: count, dtype: int64


In [161]:
# One hot encoding for nominal columns with no natural order
# pd.get_dummies creates a new binary column for each unique category
df = pd.get_dummies(df, columns=[
    "Race",
    "Marital Status",
    "Estrogen Status",
    "Progesterone Status"
], drop_first=True)

print()
print("=== ALL COLUMN NAMES AFTER FULL ENCODING ===")
print(df.columns.tolist())

print()
print("=== SHAPE AFTER ENCONDIG ===")
print(df.shape)

print()
print("=== FIRST 5 ROWS AFTER ENCODING ===")
print(df.head())


=== ALL COLUMN NAMES AFTER FULL ENCODING ===
['Age', 'T Stage', 'N Stage', '6th Stage', 'Grade', 'A Stage', 'Tumor Size', 'Regional Node Examined', 'Regional Node Positive', 'Status', 'Race_Other', 'Race_White', 'Marital Status_Married', 'Marital Status_Separated', 'Marital Status_Single ', 'Marital Status_Widowed', 'Estrogen Status_Positive', 'Progesterone Status_Positive']

=== SHAPE AFTER ENCONDIG ===
(4024, 18)

=== FIRST 5 ROWS AFTER ENCODING ===
   Age  T Stage  N Stage  6th Stage  Grade  A Stage  Tumor Size  \
0   68        1        1          1      3        0           4   
1   50        2        2          3      2        0          35   
2   58        3        3          5      2        0          63   
3   58        1        1          1      3        0          18   
4   47        2        1          2      3        0          41   

   Regional Node Examined  Regional Node Positive  Status  Race_Other  \
0                      24                       1       0       Fal

In [162]:
# Convert True/False values to 1/0 across the entire dataframe
# astype(int) converts boolean True to 1 and False to 0
df = df.astype(int)

# Marital Status_Single column has an invisible space at the end which will cause bugs
# str.strip() removes any invisible leading or trailing spaces from each column name
df.columns = df.columns.str.strip()

print("=== COLUMN NAMES AFTER FIXING TRALING SPACES")
print(df.columns.tolist())

print()
print("=== FIRST 5 ROWS AFTER FIXING TRUE/FALSE ===")
print(df.head())

=== COLUMN NAMES AFTER FIXING TRALING SPACES
['Age', 'T Stage', 'N Stage', '6th Stage', 'Grade', 'A Stage', 'Tumor Size', 'Regional Node Examined', 'Regional Node Positive', 'Status', 'Race_Other', 'Race_White', 'Marital Status_Married', 'Marital Status_Separated', 'Marital Status_Single', 'Marital Status_Widowed', 'Estrogen Status_Positive', 'Progesterone Status_Positive']

=== FIRST 5 ROWS AFTER FIXING TRUE/FALSE ===
   Age  T Stage  N Stage  6th Stage  Grade  A Stage  Tumor Size  \
0   68        1        1          1      3        0           4   
1   50        2        2          3      2        0          35   
2   58        3        3          5      2        0          63   
3   58        1        1          1      3        0          18   
4   47        2        1          2      3        0          41   

   Regional Node Examined  Regional Node Positive  Status  Race_Other  \
0                      24                       1       0           0   
1                      14   

In [163]:
# This function splits our data into training and test sets
from sklearn.model_selection import train_test_split

# x contains all columns except Status, these are the features
x = df.drop(columns=["Status"])

# x contains only the Status column, this is what we are predicting
y = df["Status"]

print("=== FEATURES SHAPE ===")
print(x.shape)

print()
print("=== TARGET SHAPE ===")
print(y.shape)

# random_state=42 means the split is reproducible, same split every time we run
# stratify=y ensures the class balance is preserved in both splits
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 80% of patients used for training the model
print()
print("=== TRAINING SET SIZE ===")
print(x_train.shape)

# 20% of patients held back for final evaluation
print()
print("=== TEST SET SIZE ===")
print(x_test.shape)

# Checking that Alive/Dead ratio is preserved after split
print()
print("=== CLASS DISTRIBUTION IN TRAINING SET ===")
print(y_train.value_counts())

#Checking that Alive/Dead ratio is preserved after split
print()
print("=== CLASS DISTRIBUTION IN TEST SET ===")
print(y_test.value_counts())

=== FEATURES SHAPE ===
(4024, 17)

=== TARGET SHAPE ===
(4024,)

=== TRAINING SET SIZE ===
(3219, 17)

=== TEST SET SIZE ===
(805, 17)

=== CLASS DISTRIBUTION IN TRAINING SET ===
Status
0    2726
1     493
Name: count, dtype: int64

=== CLASS DISTRIBUTION IN TEST SET ===
Status
0    682
1    123
Name: count, dtype: int64


In [164]:
# SMOTE = Synthetic Minority Oversampling Technique
# Import SMOTE from imbalanced-learn library
from imblearn.over_sampling import SMOTE
# Import Counter to count class distribution easily
from collections import Counter

# This is the imbalanced training set we start with
print("=== CLASS DISTRIBUTION BEFORE SMOTE ===")
print(Counter(y_train))

# Create a SMOTE object
# random_state=42 ensures reproducibility, same synthetic samples every run
smote = SMOTE(random_state=42)

# Apply SMOTE to training data only
x_train_resampled, y_train_resampled = smote.fit_resample(x_train, y_train)

print()
print("=== CLASS DISTRIBUTION AFTER SMOTE ===")
print(Counter(y_train_resampled))

print()
print("=== TRANING SET SIZE AFTER SMOTE ===")
print(x_train_resampled.shape)

=== CLASS DISTRIBUTION BEFORE SMOTE ===
Counter({0: 2726, 1: 493})

=== CLASS DISTRIBUTION AFTER SMOTE ===
Counter({0: 2726, 1: 2726})

=== TRANING SET SIZE AFTER SMOTE ===
(5452, 17)


In [165]:
# Import os to handle file paths
import os

# Create the outputs folder if it does not exist
os.makedirs("../outputs/processed_data", exist_ok=True)

# Save the resampled training features to a CSV file
x_train_resampled.to_csv("../outputs/processed_data/x_train_resampled.csv", index=False)

# Save the resampled training target to a CSV file
y_train_resampled.to_csv("../outputs/processed_data/y_train_resampled.csv", index=False)

# Save the test features to a CSV file
x_test.to_csv("../outputs/processed_data/x_test.csv", index=False)

# Save the test target to a CSV file
y_test.to_csv("../outputs/processed_data/y_test.csv", index=False)

print("=== FILES SAVED SUCCESSFULLY ===")
print("All four files saved to outputs/processed_data/ folder")
print()
print("Files saved:")
print("  outputs/processed_data/X_train_resampled.csv - training features after SMOTE")
print("  outputs/processed_data/y_train_resampled.csv - training target after SMOTE")
print("  outputs/processed_data/X_test.csv - test features, real patients only")
print("  outputs/processed_data/y_test.csv - test target, real patients only")

=== FILES SAVED SUCCESSFULLY ===
All four files saved to outputs/processed_data/ folder

Files saved:
  outputs/processed_data/X_train_resampled.csv - training features after SMOTE
  outputs/processed_data/y_train_resampled.csv - training target after SMOTE
  outputs/processed_data/X_test.csv - test features, real patients only
  outputs/processed_data/y_test.csv - test target, real patients only
